In [1]:
import pandas as pd

delivered = pd.read_parquet('../data/delivered_orders.parquet')
delivered.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_state,...,n_payment_installments,payment_type,review_score,review_creation_date,delivery_days,estimated_delivery_days,delivery_delay_days,is_late,customer_total_orders,is_repeat_customer
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,SP,...,1.0,credit_card,4.0,2017-10-11 00:00:00,8.0,15,-8.0,False,2,True
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,BA,...,1.0,boleto,4.0,2018-08-08 00:00:00,13.0,19,-6.0,False,1,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,GO,...,3.0,credit_card,5.0,2018-08-18 00:00:00,9.0,26,-18.0,False,1,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,RN,...,1.0,credit_card,5.0,2017-12-03 00:00:00,13.0,26,-13.0,False,1,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,SP,...,1.0,credit_card,5.0,2018-02-17 00:00:00,2.0,12,-10.0,False,1,False


In [2]:
repeat_rate = delivered.groupby('customer_unique_id')['is_repeat_customer'].first().mean()
print(f"Repeat purchase rate: {repeat_rate*100:.2f}%")

Repeat purchase rate: 3.19%


In [3]:
master = pd.read_parquet('../data/master_orders.parquet')
repeat_rate_all = master.groupby('customer_unique_id')['is_repeat_customer'].first().mean()
print(f"Repeat purchase rate (all customers): {repeat_rate_all*100:.2f}%")

Repeat purchase rate (all customers): 3.12%


In [4]:
delivered.groupby('is_late')['review_score'].agg(['mean', 'median', 'count'])

,mean,median,count
is_late,,,
False,4.290863,5.0,89451
True,2.272841,1.0,6381


In [5]:
first_orders = delivered.sort_values('order_purchase_timestamp').groupby('customer_unique_id').first().reset_index()
first_orders.groupby('is_late')['is_repeat_customer'].agg(['mean', 'count'])

,mean,count
is_late,,
False,0.032252,87002
True,0.027218,6356


In [6]:
from scipy import stats

late = delivered[delivered['is_late']]['review_score'].dropna()
ontime = delivered[~delivered['is_late']]['review_score'].dropna()

u_stat, p_val = stats.mannwhitneyu(late, ontime, alternative='less')

print(f"Late mean: {late.mean():.3f} (n={len(late)})")
print(f"On-time mean: {ontime.mean():.3f} (n={len(ontime)})")
print(f"Mann-Whitney U p-value: {p_val:.2e}")

Late mean: 2.273 (n=6381)
On-time mean: 4.291 (n=89451)
Mann-Whitney U p-value: 0.00e+00


In [7]:
n1, n2 = len(late), len(ontime)
r_rb = 1 - (2*u_stat)/(n1*n2)
print(f"Rank-biserial effect size: {r_rb:.3f}")

Rank-biserial effect size: 0.637


In [8]:
import statsmodels.formula.api as smf
import numpy as np

# Rebuild first_orders with the extra columns we need for the regression
first_orders_full = delivered.sort_values('order_purchase_timestamp').groupby('customer_unique_id').first().reset_index()
first_orders_full = first_orders_full.dropna(subset=['review_score', 'delivery_delay_days', 'total_payment_value', 'customer_state'])

# Collapse rare states into 'OTHER' so the model doesn't choke on tiny categories
top_states = first_orders_full['customer_state'].value_counts().nlargest(8).index
first_orders_full['state_grp'] = first_orders_full['customer_state'].where(
    first_orders_full['customer_state'].isin(top_states), 'OTHER')

first_orders_full['y'] = first_orders_full['is_repeat_customer'].astype(int)
first_orders_full['log_payment'] = np.log1p(first_orders_full['total_payment_value'])

print(f"Rows used: {len(first_orders_full)}")

Rows used: 92746


In [9]:
# H2: Does review_score alone predict repeat purchase?
m1 = smf.logit('y ~ review_score', data=first_orders_full).fit()
print(m1.summary())

Optimization terminated successfully.
         Current function value: 0.141705
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                92746
Model:                          Logit   Df Residuals:                    92744
Method:                           MLE   Df Model:                            1
Date:                Tue, 18 Aug 2026   Pseudo R-squ.:               5.294e-06
Time:                        23:35:27   Log-Likelihood:                -13143.
converged:                       True   LL-Null:                       -13143.
Covariance Type:            nonrobust   LLR p-value:                    0.7091
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -3.4313      0.063    -54.061      0.000      -3.556      -3.307
review_score     0.0054

In [10]:
m2 = smf.logit('y ~ review_score + delivery_delay_days + log_payment + n_items + C(state_grp)',
               data=first_orders_full).fit()
print(m2.summary())

Optimization terminated successfully.
         Current function value: 0.141110
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                92746
Model:                          Logit   Df Residuals:                    92733
Method:                           MLE   Df Model:                           12
Date:                Tue, 18 Aug 2026   Pseudo R-squ.:                0.004201
Time:                        23:35:51   Log-Likelihood:                -13087.
converged:                       True   LL-Null:                       -13143.
Covariance Type:            nonrobust   LLR p-value:                 4.913e-18
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                -3.1955      0.168    -19.060      0.000      -3.524      -2.

In [11]:
import numpy as np

or_per_day = np.exp(m2.params['delivery_delay_days'])
effect_30d = or_per_day**30
print(f"Odds ratio per day of delay: {or_per_day:.4f}")
print(f"Odds multiplier for 30 days later delivery: {effect_30d:.3f}")
print(f"Relative odds reduction: {(1-effect_30d)*100:.1f}%")

Odds ratio per day of delay: 0.9908
Odds multiplier for 30 days later delivery: 0.758
Relative odds reduction: 24.2%


In [12]:
products = pd.read_csv('../data/olist_products_dataset.csv')
items = pd.read_csv('../data/olist_order_items_dataset.csv')

# Map each order to its (first) product category
order_category = items.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
order_category = order_category.groupby('order_id')['product_category_name'].first().reset_index()

# Attach category onto first_orders_full (built earlier for the regression)
first_orders_cat = first_orders_full.merge(order_category, on='order_id', how='left')
first_orders_cat['product_category_name'] = first_orders_cat['product_category_name'].fillna('unknown')

print(f"Rows: {len(first_orders_cat)}")

Rows: 92746


In [13]:
category_repeat = first_orders_cat.groupby('product_category_name')['is_repeat_customer'].agg(['mean', 'count'])
category_repeat = category_repeat[category_repeat['count'] >= 200].sort_values('mean', ascending=False)
category_repeat.head(15)

,mean,count
product_category_name,,
eletrodomesticos,0.092262,672
fashion_bolsas_e_acessorios,0.061055,1687
fashion_calcados,0.058559,222
climatizacao,0.051724,232
moveis_decoracao,0.047231,5886
cama_mesa_banho,0.046495,8646
bebidas,0.044776,268
casa_conforto,0.042135,356
moveis_sala,0.041667,384


In [14]:
print("Lowest repeat-rate categories (min 200 orders):")
print(category_repeat.tail(10))
print()
print(f"Category repeat rate range: {category_repeat['mean'].min()*100:.2f}% to {category_repeat['mean'].max()*100:.2f}%")
print(f"Category repeat rate std dev: {category_repeat['mean'].std()*100:.2f} percentage points")

Lowest repeat-rate categories (min 200 orders):
                                       mean  count
product_category_name                             
automotivo                         0.021969   3687
malas_acessorios                   0.020408    980
eletroportateis                    0.020408    588
instrumentos_musicais              0.020305    591
cool_stuff                         0.020023   3446
eletrodomesticos_2                 0.018605    215
consoles_games                     0.018237    987
construcao_ferramentas_iluminacao  0.018182    220
eletronicos                        0.017580   2446
livros_tecnicos                    0.007968    251

Category repeat rate range: 0.80% to 9.23%
Category repeat rate std dev: 1.48 percentage points
